In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import (
    clean_column_names,
    clean_total_charges,
    encode_Yes_No_features,
    prepare_target,
    drop_unused_columns,
    encode_binary_features,
    encode_categorical_features
)

In [ ]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df.head()
df.info()
df.describe()
df.shape

In [ ]:
df["Churn"].value_counts(normalize=True)

In [ ]:
plt.hist(df["MonthlyCharges"])
plt.show()

In [ ]:
df.groupby("Contract")["Churn"].value_counts(normalize=True)

In [ ]:
pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
)

In [ ]:
plt.hist(df[df["Churn"]=="Yes"]["tenure"], bins=30)
plt.show()

In [ ]:
df["TotalCharges"].dtype

In [ ]:
pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
)

In [ ]:
pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
)

In [ ]:
pd.crosstab(
    df["Partner"],
    df["Churn"],
    normalize="index"
)

In [ ]:
df = drop_unused_columns(df)

In [ ]:
df = clean_column_names(df)
df = clean_total_charges(df)
df = prepare_target(df)
df = encode_binary_features(df)
df = encode_categorical_features(df)

In [ ]:
df.dtypes

In [ ]:
df.select_dtypes(include="object").columns

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [ ]:
X.shape
y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled.mean(axis=0)
X_train_scaled.std(axis=0)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(
    random_state=42,
    max_iter=1000
)
model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(
    y_test,
    y_pred
)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(
    y_test,
    y_pred
)

In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(
    X_test_scaled
)[:,1]

roc_auc_score(
    y_test,
    y_prob
)

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coef_df = coef_df.sort_values(
    by="Coefficient",
    ascending=False
)

coef_df

In [ ]:
coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()

coef_df= coef_df.sort_values(
    by="AbsCoefficient",
    ascending=False
)

coef_df

In [ ]:
corr = df[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]
].corr()

corr

In [ ]:
df[df["tenure"]==0]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model=RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

In [ ]:
rf_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
accuracy_score(
    y_test,
    y_pred_rf
)

In [ ]:
print(
    classification_report(
    y_test,
    y_pred_rf
    )
)

In [ ]:
confusion_matrix(
    y_test,
    y_pred_rf
)

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

roc_auc_score(
    y_test,
    y_prob_rf
)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
)